# Feature Extraction 

Michael Mommert, Stuttgart University of Applied Sciences, 2026

This Notebook provides an introduction into feature extraction techniques for different data modalities.

In [ ]:
%pip install numpy \
    pandas \
    matplotlib \
    pillow \
    scikit-image \
    scikit-learn \
    nltk

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Quantitative Data

Consider the following DataFrame. Create a new DataFrame that contains the temperature in units of Kelvin and the rain amount in units of inches.

In [ ]:
df = pd.DataFrame({
    'temp_C': [-0.3, 0.4, 3.9, 7.4, 12.0, 15.0, 17.2, 16.8, 13.1, 9.1, 3.7, 0.8],
    'rain_mm': [59, 57, 84, 100, 143, 153, 172, 164, 135, 89, 88, 80]},
    index=['jan', 'feb', 'mar', 'apr', 'may', 'jun', 
           'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])
df

**Exercise**: Convert the temperature values into units of Kelvins (0K = -273.16C).

In [ ]:
# use this cell for the exercise

**Exercise**: Convert the rain amount from millimeters to inches (1 inch = 25.4mm). 

In [ ]:
# use this cell for the exercise

**Exercise**: Create a new DataFrame (`df2`) that contains both the temperature in Kelvin and the rain amount in inches.

In [ ]:
# use this cell for the exercise

## Qualitative Data

Consider the following multi-class feature: 

In [ ]:
conditions = ['very bad', 'average', 'good', 'bad', 'very good', 'average', 'bad', 'very bad', 'very good', 'average']

We will now **label-encode** this feature. 

In the first step, we have to identify the different unique labels. This can be done with a set:

In [ ]:
list(set(conditions))

(Note the order of elements might differ)

We see that the following labels are present: `very bad`, `bad`, `average`, `good`, `very good`

Naturally, this is an ordinal (ranked) qualitative feature. We can assign each label a ranked discrete numerical label. This can be done using a dictionary: 

In [ ]:
conditions_lookuptable = {'very bad': 0, 'bad': 1, 'average': 2, 'good': 3, 'very good': 4}

In the final step, we translate each label into the corresponding numerical feature:

In [ ]:
conditions_num = [conditions_lookuptable[c] for c in conditions]
conditions_num

We will now consider a nominal qualitative feature:

In [ ]:
fruits = ['apple', 'orange', 'pear', 'orange', 'melon', 'apple', 'lemon', 'pear', 'melon', 'apple']

Again, we extract the unique classes:

In [ ]:
list(set(fruits))

Since the classes are unranked, we should use **one-hot label encoding** in this case. This can be easily achieved with existing functionality from the `scikit-learn` module: 

In [ ]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder()

fruits_onehot = enc.fit_transform(np.array(fruits).reshape(-1, 1)).toarray()
fruits_onehot

In order to interpret this array, we need to know the different classes:

In [ ]:
enc.categories_

Let's consider a more complex example involving multiple labels:

In [ ]:
pizza = [['tomatoes', 'cheese'],
         ['tomatoes', 'cheese', 'peperoni'],
         ['tomatoes', 'cheese', 'prosciutto'],
         ['tomatoes', 'cheese', 'peperoni', 'mushrooms'],
         ['tomatoes', 'cheese', 'prosciutto', 'peperoni', 'mushrooms'],
         ['pesto']
         ]

In order to encode this list of pizza toppings, we need a different method (which also performs one-hot encoding, but allows for a varying number of labels per sample), which is called `MultiLabelBinarizer`:

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

enc = MultiLabelBinarizer()

enc.fit_transform(pizza)

Again, the different labels are sorted in a specific order. This order is revealed by the encoder:

In [ ]:
enc.classes_

## Image Data

We download an image and extract different image features in the following.

In [ ]:
!curl -O https://raw.githubusercontent.com/Hochschule-fuer-Technik-Stuttgart/teaching-mommert/main/dataprocessing/featureextraction/IMG_20230622_085147.jpg

In [ ]:
from PIL import Image

# read image
img = np.array(Image.open('IMG_20230622_085147.jpg').convert('RGB'))

# display image
plt.imshow(img)

### Channel Histograms

A channel histogram extracts the pixel value distribution of each image channel or band. 

**Exercise**: Extract the R, G, and B band information into arrays `r`, `g` and `b`, respectively.

In [ ]:
# use this cell for the exercise

**Exercise**: Create and plot the channel histograms.

In [ ]:
# use this cell for the exercise

**Exercise**: Combine the three channel histograms into a single vector by concatenating their individual vectors.

In [ ]:
# use this cell for the exercise

### Canny Edges

The Canny method extracts edges from an image. 

**Exercise**: Use the method (implemented in the scikit-image module) to extract edges from the provided image and plot the resulting array.

In [ ]:
from skimage.feature import canny

# use this cell for the exercise

### SIFT

We extract SIFT features from our image:

In [ ]:
from skimage.feature import SIFT

img_grey = img[::4, ::4, 1]  # extract G band and rescale by factor of 4

sift = SIFT()
sift.detect_and_extract(img_grey)

keypoints = sift.keypoints
descriptors = sift.descriptors

plt.imshow(img_grey, cmap='Greys_r')
plt.scatter(keypoints[:, 1], keypoints[:, 0], s=5)

Let's have a look at one of the descriptors:

In [ ]:
descriptors[0]

**Exercise**: We rotate the image by 180 degrees and extract SIFT keypoints again. Can we match the keypoints from both images based on their descriptors? Identify the 5 best-matching keypoints from both images based on the euclidean distance and plot their positions in both images.

In [ ]:
img2 = Image.fromarray(img).rotate(180)

# use this cell for the exercise

## Text Data

We download the first chapter of the book "Frankenstein" and experiment with some feature extraction methods for text data.

In [ ]:
!curl -O https://raw.githubusercontent.com/Hochschule-fuer-Technik-Stuttgart/teaching-mommert/main/dataprocessing/featureextraction/frankenstein_chapter1.txt

In [ ]:
# open text file
with open("frankenstein_chapter1.txt", "r") as f:
    data = f.readlines()

# extract all linebreaks from the text file
text = ""
for line in data:
    text += line.replace('\n', ' ')
text

**Exercise**: Use `nltk.word_tokenize` to tokenize the text.

In [ ]:
import nltk
nltk.download('punkt')

# use this cell for the exercise
#tokens = ...

We can tag the tokens using a tagger:

In [ ]:
nltk.download('averaged_perceptron_tagger')

nltk.pos_tag(tokens)

We can also use a stemmer to find the word stems of our tokens.

In [ ]:
from nltk.stem.porter import *

stemmer = PorterStemmer()

for w in tokens[:100]:
    print(w, stemmer.stem(w))

**Exercise**: Based on the word stems, find the 10 most common words.

In [ ]:
# use this cell for the exercise